In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from feature_engineering.order_flow_pipeline import OrderFlowFeatureEngineering
from feature_engineering.order_flow import L2QFeatureEngineering, TradeFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Run single pipeline with both trades and level2q data types
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades', 'level2q']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer(),
        feature_engineering=TradeFeatureEngineering(bar_duration_ms=BAR_DURATION_MS)
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=5,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=5,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

pipeline = Pipeline(config)
results = pipeline.run(slice(10))

In [ ]:
# Verify results
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

# Show results by data type
trades_results = [r for r in results if 'trades' in r.get('output_path', '')]
l2q_results = [r for r in results if 'level2q' in r.get('output_path', '')]

print(f"\nTrades results: {len(trades_results)}")
for result in trades_results[:3]:
    if result['message'] == 'success':
        print(f"✓ trades: {result.get('row_count', 0):,} rows -> {result['output_path']}")
    else:
        print(f"✗ Failed: {result['message'][:100]}")

print(f"\nLevel2Q results: {len(l2q_results)}")
for result in l2q_results[:3]:
    if result['message'] == 'success':
        print(f"✓ level2q: {result.get('row_count', 0):,} rows -> {result['output_path']}")
    else:
        print(f"✗ Failed: {result['message'][:100]}")